<table  align="left" width="100%"> <tr>
        <td  style="background-color:#ffffff;"><a href="https://qworld.net" target="_blank"><img src="../images/qworld/qworld.jpg" width="35%" align="left"></a></td>
        <td  align="right" style="background-color:#ffffff;vertical-align:bottom;horizontal-align:right">
            prepared by Adam Glos and Özlem Salehi Köken</a>
        </td>        
</tr></table>

# <font color="blue"> Solutions for </font> Max-Cut Problem

<a id="task1"></a>
### Task 1
    
Let's implement the idea given above. We have a graph with 4 vertices, and so we have a circuit with 4 qubits. 

Represent the following coloring of the given graph in the quantum circuit.

<img src="../images/graphcolor1.png" width="25%" align="center">

<h3> Solution </h3>

In [3]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

qc = QuantumCircuit(4, 4)
qc.x(1)
qc.x(2)

qc.measure(range(4), range(4))
job = AerSimulator().run(qc, shots=1)
counts = job.result().get_counts(qc)
print(counts)

for qubit in range(4):
    if list(counts.keys())[0][qubit] == '0':
        print("Node",qubit, "is red")
    else:
        print("Node",qubit, "is blue")

{'0110': 1}
Node 0 is red
Node 1 is blue
Node 2 is blue
Node 3 is red


<a id="task2"></a>
### Task 2

For the given graph below, implement the first two steps of the oracle described above. 

The first four qubits are used for vertices.

The next three qubits are used for edges.

The last qubit is used for the output.

Remark that the last qubit should be in state $ \ket{1} $ if the endpoints of all the edges are colored using a different color and $\ket{0}$ otherwise.

Test your implementation with different colorings.

<img src="../images/bipartite.png" width="25%" align="center">

<h3> Solution </h3>

In [7]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import MCXGate

qc = QuantumCircuit(8, 1)

# All end points have different colors
qc.x(0)
qc.x(1)

# Not all end points have different colors
# qc.x(0)

# check 0-2 edge and store at 4th qubit
qc.cx(0, 4)
qc.cx(2, 4)
# check 0-3 edge and store at 5th qubit
qc.cx(0, 5)
qc.cx(3, 5)
# check 1-3 edge and store at 6th qubit
qc.cx(1, 6)
qc.cx(3, 6)

# check all edge qubits
qc.mcx([4,5,6], 7)
qc.measure(7, 0)

job = AerSimulator().run(qc, shots=1)
counts = job.result().get_counts(qc)
print(counts)

if list(counts.keys())[0][0] == '1':
    print("All end points have different colors (graph is bipartite)")
else:
    print("Not all end points have different colors (graph is not bipartite)")

{'1': 1}
All end points have different colors (graph is bipartite)


<a id="task3"></a>
### Task 3 

For the given graphs below, iterate Grover’s search algorithm 2 steps to find the correct colorings. (There are indeed $k=2$ possible colorings, and so the oracle should be applied $\frac{\pi}{4}\sqrt{\frac{2^4}{k}}\approx 2$ times.) 

You will use nine qubits: 4 for vertices, 4 for edges, and 1 for the output qubit.

The diffusion operator is provided below.

Observe which outcomes have the higher frequencies.


<img src="../images/completebipartite.png" width="25%" align="center">

<h3> Solution </h3>

In [ ]:
from qiskit import QuantumCircuit, transpile    
from qiskit.circuit.library import ZGate


def edge_check(a, b, c):
    yield CX(qq[a], qq[c])
    yield CX(qq[b], qq[c])

def oracle_computation():
    temp_qc = QuantumCircuit(9)
    # check 0-2 edge and store at 4th qubit
    temp_qc.cx(0, 4)
    temp_qc.cx(2, 4)
    # check 0-3 edge and store at 5th qubit
    temp_qc.cx(0, 5)
    temp_qc.cx(3, 5)
    # check 1-2 edge and store at 6th qubit
    temp_qc.cx(1, 6)
    temp_qc.cx(2, 6)
    # check 1-3 edge and store at 7th qubit
    temp_qc.cx(1, 7)
    temp_qc.cx(3, 7)

    # check all edge qubits
    temp_qc.mcx([4,5,6,7], 8)

    return temp_qc
    
def oracle(qc):
    qc.append(oracle_computation(), range(9))
    qc.z(8)
    qc.append(oracle_computation().inverse(), range(9))

def grover_diffusion(qc):
    qc.h(range(4))
    qc.x(range(4))
    qc.append(ZGate().control(3), range(4))
    qc.x(range(4))
    qc.h(range(4))

qc = QuantumCircuit(9, 4)
for i in range(4):
    qc.h(i)
for i in range(2):
    oracle(qc)
    grover_diffusion(qc)
qc.measure(range(4), range(4))
tqc = transpile(qc, AerSimulator())
job = AerSimulator().run(tqc, shots=1024)
counts = job.result().get_counts(qc)
print(counts)

{'1010': 1, '1001': 7, '1110': 4, '0110': 3, '1011': 3, '0011': 468, '0001': 5, '1100': 486, '0100': 7, '0000': 3, '0010': 7, '0101': 6, '1111': 7, '1101': 5, '1000': 7, '0111': 5}


Note that the states '0011' and '1100' are the most commonly observed states.

In [14]:
print("Probability of measuring 0011: ", counts.get('0011')/1024)
print("Probability of measuring 1100: ", counts.get('1100')/1024)

Probability of measuring 0011:  0.45703125
Probability of measuring 1100:  0.474609375


<a id="task4"></a>
### Task 4

Repeat Task 3 for the following graph.

Is it possible to color the following graph so that all edges have endpoints with different colors? If not, what would you say in advance about the frequencies of the outcomes?


<img src="../images/nonbipartite.png" width="25%" align="center">

<h3> Solution </h3>

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import ZGate

def oracle_computation():
    temp_qc = QuantumCircuit(9)
    # check 0-1 edge and store at 4th qubit
    qc.cx(0, 4)
    qc.cx(1, 4)
    # check 0-2 edge and store at 5th qubit
    qc.cx(0, 5)
    qc.cx(2, 5)
    # check 0-3 edge and store at 6th qubit
    qc.cx(0, 6)
    qc.cx(3, 6)
    # check 1-3 edge and store at 7th qubit
    qc.cx(1, 7)
    qc.cx(3, 7)

    # check all edge qubits
    qc.mcx([4,5,6,7], 8)

    return temp_qc

def oracle(qc):
    qc.append(oracle_computation(), range(9))
    qc.z(8)
    qc.append(oracle_computation().inverse(), range(9))
    
def grover_diffusion(qc):
    qc.h(range(4))
    qc.x(range(4))
    qc.append(ZGate().control(3), range(4))
    qc.x(range(4))
    qc.h(range(4))

qc = QuantumCircuit(9, 4)
for i in range(4):
    qc.h(i) 
for i in range(2):
    oracle(qc)
    grover_diffusion(qc)
   
# we are only intertested in outputs of first 4 qubits
qc.measure(range(4), range(4))

tqc = transpile(qc, AerSimulator())
job = AerSimulator().run(tqc, shots=1024)
counts = job.result().get_counts(qc)
print(counts)

{'1011': 50, '1010': 61, '1000': 49, '1001': 60, '0110': 62, '0011': 79, '0001': 54, '1111': 78, '1101': 66, '0000': 69, '0010': 55, '0100': 78, '1100': 59, '1110': 69, '0111': 71, '0101': 64}


We observe that all strings have almost the same occurence. The reason is that there is no coloring where all the endpoints have different colors.